# 001 — Model Shootout: XGBoost vs Random Forest vs Logistic Regression vs HistGradientBoosting

A comprehensive comparison spanning the full spectrum of classification models:
from a simple linear baseline (Logistic Regression) through classical ensembles
(Random Forest) to modern gradient boosting (XGBoost, HistGradientBoosting).
Each model is tuned with Optuna, evaluated with threshold analysis, and
examined through feature importance and SHAP values.

**Lifecycle stage:** seedling (model-garden)

All code is self-contained in this notebook.

In [ ]:
# ---------------------------------------------------------------------------
# Papermill parameters  (this cell is tagged "parameters")
# ---------------------------------------------------------------------------

# Data loading
feature_paths: list[str] = []          # local or gs:// URIs; empty -> synthetic
target_col: str = 'target'

# Splitting
test_size: float = 0.2
random_state: int = 42

# Optuna (per model)
optuna_n_trials: int = 20
optuna_timeout_s: int | None = None
optimize_metric: str = 'f1'            # 'f1', 'precision', 'recall'
threshold_grid: list[float] = [round(x * 0.05, 2) for x in range(1, 20)]

# Outputs
metrics_json_path: str = 'outputs/metrics/metrics.json'
plots_dir: str = 'outputs/plots'
executed_notebook_path: str | None = None

# SHAP / feature importance
shap_sample_size: int = 500
enable_shap: bool = True
enable_feature_importance: bool = True

In [ ]:
# ---------------------------------------------------------------------------
# Imports
# ---------------------------------------------------------------------------
import json
import os
import time
import warnings
from datetime import datetime, timezone
from pathlib import Path

warnings.filterwarnings('ignore')
os.environ['PYTHONWARNINGS'] = 'ignore'

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
import joblib
import xgboost as xgb
from sklearn.datasets import make_classification
from sklearn.ensemble import (
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    auc,
    average_precision_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.calibration import calibration_curve
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from scipy.stats import spearmanr
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)

for d in ['outputs/runs', 'outputs/plots', 'outputs/models', 'outputs/metrics']:
    Path(d).mkdir(parents=True, exist_ok=True)

print(f'Run started at {datetime.now(timezone.utc).isoformat()}')

## 1 — Meet the Contenders

This comparison spans four fundamentally different approaches to
classification. Understanding *why* they differ matters more than knowing
which one "wins" — the right model depends on your constraints.

### XGBoost (Gradient Boosting)

XGBoost builds an ensemble of weak decision trees **sequentially**. Each new
tree corrects the errors of the previous ones by fitting the gradient of the
loss function. Key characteristics:

- **Sequential learning** — trees depend on each other; can't fully parallelize
- **Regularization** — L1 (`reg_alpha`), L2 (`reg_lambda`), and minimum loss
  reduction (`gamma`) prevent overfitting
- **`enable_categorical=True`** — native categorical support via optimal
  partitioning (requires pandas `Categorical` dtype)
- **Early stopping** — monitors validation loss, stops when no improvement
- **Multiple importance types** — `weight`, `gain`, `cover`

Best for: maximum predictive performance with moderate interpretability.

### Random Forest (Bagging)

Random Forest builds many **independent** decision trees in parallel, each
on a bootstrap sample of the data, and averages their predictions. This is
*bagging*, not boosting:

- **Parallel training** — trees are independent; trivially parallelizable
  with `n_jobs=-1`
- **OOB score** — each tree validates on the ~36.8% of samples *not* in its
  bootstrap sample, giving a free generalization estimate without a holdout set
- **Robust to hyperparameters** — rarely overfits badly, works well
  out-of-the-box with default settings
- **No native categorical support** — needs encoding (ordinal or one-hot)
- **Feature importance** — Gini impurity (default) or permutation-based

Best for: robust predictions with minimal tuning, when you want a strong
baseline that's hard to mess up.

### Logistic Regression (Linear Model)

Logistic Regression is a linear model — it learns a single weight per feature
and applies a sigmoid to produce probabilities. Completely different from
tree ensembles:

- **Linear decision boundary** — can only model linear relationships (or
  interactions if you engineer them)
- **Coefficients are interpretable** — each coefficient tells you the
  direction and magnitude of a feature's effect on the log-odds
- **Requires preprocessing** — needs feature scaling (StandardScaler) and
  categorical encoding; sensitive to feature magnitudes
- **Regularization** — L1 (sparse coefficients), L2 (shrink coefficients),
  or ElasticNet (both)
- **Fast training** — typically trains in seconds even on large datasets
- **Well-calibrated probabilities** — probabilities are meaningful out of the
  box (no calibration needed)

Best for: interpretable baselines, understanding feature effects, regulatory
contexts where you need to explain every prediction.

### HistGradientBoostingClassifier (sklearn's Modern Boosting)

sklearn's histogram-based gradient boosting, inspired by LightGBM. It's
the **sklearn-native** answer to XGBoost/LightGBM:

- **Histogram-based splits** — bins continuous features into 256 bins for
  fast split finding
- **Native categorical support** — via `categorical_features` parameter with
  ordinal-encoded integers
- **Native missing value handling** — handles NaN without imputation
- **Early stopping** — built-in via `early_stopping=True` with internal
  validation split
- **No external dependencies** — pure sklearn; if you already use sklearn,
  no new library needed
- **`max_leaf_nodes`** — controls tree complexity (similar to LightGBM's
  `num_leaves`)

Best for: when you want boosting performance without adding external
dependencies, or when your data has missing values.

### Comparison at a Glance

| Aspect | XGBoost | Random Forest | Logistic Reg. | HistGBM |
|---|---|---|---|---|
| Model type | Boosting | Bagging | Linear | Boosting |
| Trees | Sequential | Parallel | N/A | Sequential |
| Categorical support | Native | Needs encoding | Needs encoding | Native |
| Feature scaling needed? | No | No | Yes | No |
| Handles non-linearity? | Yes | Yes | No | Yes |
| Interpretability | SHAP | SHAP / Gini | Coefficients | SHAP |
| Training speed | Moderate | Fast (parallel) | Very fast | Fast |
| Overfitting risk | Moderate | Low | Low (linear) | Moderate |
| Hyperparameter sensitivity | High | Low | Low | Moderate |
| Deployment complexity | External lib | sklearn | sklearn | sklearn |

## 2 — Data Loading

In [ ]:
# ---------------------------------------------------------------------------
# Data loading — synthetic dataset with numeric + categorical features
# ---------------------------------------------------------------------------

def _read_file(path: str) -> pl.DataFrame:
    p = path.strip()
    if p.endswith('.parquet') or p.endswith('.pq'):
        return pl.read_parquet(p)
    return pl.read_csv(p)

if feature_paths:
    dfs = [_read_file(p) for p in feature_paths]
    df = pl.concat(dfs, how='vertical_relaxed')
    print(f'Loaded {len(feature_paths)} file(s): {df.shape[0]:,} rows x {df.shape[1]} cols')
else:
    print('No feature_paths provided — generating synthetic dataset.')
    n_samples = 8_000
    rng = np.random.RandomState(random_state)

    X_num, y = make_classification(
        n_samples=n_samples, n_features=15, n_informative=10,
        n_redundant=3, n_classes=2, weights=[0.7, 0.3],
        flip_y=0.03, random_state=random_state,
    )

    # Categorical features with varying cardinality
    region = rng.choice(['north', 'south', 'east', 'west'], size=n_samples)
    tier = rng.choice(
        ['bronze', 'silver', 'gold', 'platinum'],
        size=n_samples, p=[0.4, 0.3, 0.2, 0.1],
    )
    channel = rng.choice(
        ['web', 'mobile', 'api', 'partner', 'direct'], size=n_samples,
    )

    # Make categoricals somewhat predictive
    boost = ((tier == 'platinum') | (tier == 'gold')) & (rng.random(n_samples) > 0.5)
    y[boost] = 1
    suppress = (region == 'north') & (rng.random(n_samples) > 0.7)
    y[suppress] = 0

    cols = {f'feat_{i:02d}': X_num[:, i] for i in range(X_num.shape[1])}
    cols['region'] = region
    cols['tier'] = tier
    cols['channel'] = channel
    cols[target_col] = y.astype(int)
    df = pl.DataFrame(cols)

# Identify column types
all_feature_cols = [c for c in df.columns if c != target_col]
_NUMERIC_TYPES = {
    pl.Float32, pl.Float64, pl.Int8, pl.Int16, pl.Int32, pl.Int64,
    pl.UInt8, pl.UInt16, pl.UInt32, pl.UInt64,
}
num_cols = [c for c in all_feature_cols if df[c].dtype in _NUMERIC_TYPES]
cat_cols = [c for c in all_feature_cols if c not in num_cols]
feature_names = num_cols + cat_cols

print(f'DataFrame: {df.shape[0]:,} rows x {df.shape[1]} cols')
print(f'Numeric features ({len(num_cols)}): {num_cols[:5]}{"..." if len(num_cols) > 5 else ""}')
print(f'Categorical features ({len(cat_cols)}): {cat_cols}')

## 3 — EDA

In [ ]:
# ---------------------------------------------------------------------------
# Schema, target distribution, histograms, categorical distributions
# ---------------------------------------------------------------------------
print('Schema:')
for name, dtype in zip(df.columns, df.dtypes):
    null_ct = df[name].null_count()
    print(f'  {name:30s}  {str(dtype):15s}  nulls={null_ct}')

print(f'\nTarget distribution ({target_col}):')
print(df[target_col].value_counts().sort(target_col))

# Histograms (first 8 numeric features)
plot_cols = num_cols[:8]
if plot_cols:
    n = len(plot_cols)
    ncols = min(4, n)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3 * nrows))
    axes_flat = np.array(axes).flatten() if n > 1 else [axes]
    for i, col in enumerate(plot_cols):
        axes_flat[i].hist(df[col].drop_nulls().to_numpy(), bins=40, edgecolor='white')
        axes_flat[i].set_title(col, fontsize=10)
    for j in range(i + 1, len(axes_flat)):
        axes_flat[j].set_visible(False)
    fig.tight_layout()
    fig.savefig(f'{plots_dir}/eda_histograms.png', dpi=120)
    plt.show()

# Categorical distributions
if cat_cols:
    n_cat = len(cat_cols)
    fig, axes = plt.subplots(1, n_cat, figsize=(5 * n_cat, 4))
    if n_cat == 1:
        axes = [axes]
    for ax, col in zip(axes, cat_cols):
        vc = df[col].value_counts().sort(col)
        ax.bar(vc[col].to_list(), vc['count'].to_list(), edgecolor='white')
        ax.set_title(col)
        ax.tick_params(axis='x', rotation=45)
    fig.tight_layout()
    fig.savefig(f'{plots_dir}/eda_categoricals.png', dpi=120)
    plt.show()

## 4 — Train / Test Split & Data Preparation

Each model needs the data in a different format. We create four
representations from one split:

- **XGBoost:** pandas DataFrame with `Categorical` dtype columns
  (`enable_categorical=True`)
- **Random Forest:** numpy array with ordinal-encoded categoricals
  (no native categorical support)
- **Logistic Regression:** numpy array with ordinal-encoded categoricals +
  StandardScaler (linear models need scaled features)
- **HistGradientBoosting:** numpy array with ordinal-encoded categoricals +
  `categorical_features` indices (native support via integer codes)

We use OrdinalEncoder for RF, LogReg, and HistGBM so all four models share
the same feature space (18 features), making SHAP comparison meaningful.

In [ ]:
# ---------------------------------------------------------------------------
# Train / test split and framework-specific data preparation
# ---------------------------------------------------------------------------
X_all_pd = df.select(feature_names).to_pandas()
y_all = df[target_col].to_numpy().astype(int)

X_train_pd, X_test_pd, y_train, y_test = train_test_split(
    X_all_pd, y_all, test_size=test_size, random_state=random_state, stratify=y_all,
)
X_train_pd = X_train_pd.reset_index(drop=True)
X_test_pd = X_test_pd.reset_index(drop=True)

# Class imbalance
n_neg = int((y_train == 0).sum())
n_pos = int((y_train == 1).sum())
scale_pos_weight = n_neg / n_pos

print(f'Train: {X_train_pd.shape[0]:,}  |  Test: {X_test_pd.shape[0]:,}')
print(f'Train target rate: {y_train.mean():.3f}  |  Test target rate: {y_test.mean():.3f}')
print(f'scale_pos_weight: {scale_pos_weight:.4f}')

# -- Ordinal encode categoricals (shared by RF, LogReg, HistGBM) ----------
oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
oe.fit(X_train_pd[cat_cols])

X_train_enc = X_train_pd.copy()
X_test_enc = X_test_pd.copy()
X_train_enc[cat_cols] = oe.transform(X_train_pd[cat_cols])
X_test_enc[cat_cols] = oe.transform(X_test_pd[cat_cols])
X_train_np = X_train_enc.values.astype(np.float64)
X_test_np = X_test_enc.values.astype(np.float64)

cat_feature_indices = [feature_names.index(c) for c in cat_cols]
print(f'\nOrdinal-encoded categorical indices: {cat_feature_indices}')
for i, c in zip(cat_feature_indices, cat_cols):
    print(f'  {c}: {list(oe.categories_[cat_cols.index(c)])}')

# -- StandardScaler for Logistic Regression --------------------------------
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_np)
X_test_scaled = scaler.transform(X_test_np)
print(f'\n[LogReg] Scaled features (mean~0, std~1)')

# -- XGBoost: pandas Categorical dtype + enable_categorical ----------------
X_train_xgb = X_train_pd.copy()
X_test_xgb = X_test_pd.copy()
for c in cat_cols:
    cat_type = pd.CategoricalDtype(categories=sorted(X_train_pd[c].unique()))
    X_train_xgb[c] = X_train_xgb[c].astype(cat_type)
    X_test_xgb[c] = X_test_xgb[c].astype(cat_type)
print(f'[XGBoost] Converted {len(cat_cols)} columns to pandas Categorical dtype')

## 5 — XGBoost Deep Dive

XGBoost is the external heavy-hitter. With `enable_categorical=True` and
`tree_method='hist'`, it handles categorical features natively through
optimal partitioning — no one-hot encoding needed.

Key hyperparameters tuned:
- `n_estimators`, `max_depth`, `learning_rate` — tree structure
- `reg_alpha`, `reg_lambda`, `gamma` — explicit regularization (L1, L2,
  min split loss)
- `subsample`, `colsample_bytree` — stochastic training
- `min_child_weight` — minimum sum of instance weight in a leaf
- `scale_pos_weight` — handles class imbalance

In [ ]:
# ---------------------------------------------------------------------------
# XGBoost — Optuna hyperparameter tuning + final model
# ---------------------------------------------------------------------------
_METRIC_FN = {
    'f1': f1_score,
    'precision': precision_score,
    'recall': recall_score,
}


def xgb_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 1500),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'gamma': trial.suggest_float('gamma', 1e-3, 5.0, log=True),
        'scale_pos_weight': scale_pos_weight,
    }
    metric_fn = _METRIC_FN[optimize_metric]
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=random_state)
    fold_scores = []
    for tr_idx, vl_idx in skf.split(X_train_xgb, y_train):
        model = xgb.XGBClassifier(
            **params, enable_categorical=True, tree_method='hist',
            eval_metric='logloss', random_state=random_state,
            early_stopping_rounds=50, verbosity=0,
        )
        model.fit(
            X_train_xgb.iloc[tr_idx], y_train[tr_idx],
            eval_set=[(X_train_xgb.iloc[vl_idx], y_train[vl_idx])], verbose=False,
        )
        proba = model.predict_proba(X_train_xgb.iloc[vl_idx])[:, 1]
        best = max(
            metric_fn(y_train[vl_idx], (proba >= t).astype(int), zero_division=0)
            for t in threshold_grid
        )
        fold_scores.append(best)
    return float(np.mean(fold_scores))


print('Tuning XGBoost...')
xgb_study = optuna.create_study(direction='maximize')
xgb_study.optimize(xgb_objective, n_trials=optuna_n_trials, timeout=optuna_timeout_s)

xgb_best_params = xgb_study.best_params
print(f'Best CV {optimize_metric}: {xgb_study.best_value:.4f}')
print(f'Best params: {json.dumps(xgb_best_params, indent=2)}')

# Train final XGBoost model
print('\nTraining final XGBoost model...')
xgb_model = xgb.XGBClassifier(
    **xgb_best_params, enable_categorical=True, tree_method='hist',
    scale_pos_weight=scale_pos_weight, eval_metric='logloss',
    random_state=random_state, early_stopping_rounds=100, verbosity=0,
)
xgb_t0 = time.time()
xgb_model.fit(
    X_train_xgb, y_train,
    eval_set=[(X_train_xgb, y_train), (X_test_xgb, y_test)], verbose=False,
)
xgb_train_time = time.time() - xgb_t0

xgb_model.save_model('outputs/models/xgboost_model.ubj')
print(f'\nXGBoost training time: {xgb_train_time:.2f}s')
print(f'Best iteration: {xgb_model.best_iteration}')
print(f'Model saved -> outputs/models/xgboost_model.ubj')

In [ ]:
# ---------------------------------------------------------------------------
# XGBoost — bells & whistles
# ---------------------------------------------------------------------------

# -- Learning curve --------------------------------------------------------
xgb_evals = xgb_model.evals_result()
xgb_eval_keys = list(xgb_evals.keys())
xgb_metric_key = list(xgb_evals[xgb_eval_keys[0]].keys())[0]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(xgb_evals[xgb_eval_keys[0]][xgb_metric_key], label='Train', alpha=0.8)
ax.plot(xgb_evals[xgb_eval_keys[1]][xgb_metric_key], label='Validation', alpha=0.8)
if xgb_model.best_iteration is not None:
    ax.axvline(xgb_model.best_iteration, color='red', linestyle='--', alpha=0.7,
               label=f'Best iteration ({xgb_model.best_iteration})')
ax.set_xlabel('Iteration')
ax.set_ylabel(xgb_metric_key)
ax.set_title('XGBoost — Learning Curve')
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(f'{plots_dir}/xgb_learning_curve.png', dpi=120)
plt.show()

# -- Multiple feature importance types ------------------------------------
if enable_feature_importance:
    imp_types = ['weight', 'gain', 'cover']
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    for ax, imp_type in zip(axes, imp_types):
        booster = xgb_model.get_booster()
        scores = booster.get_score(importance_type=imp_type)
        if scores:
            sorted_scores = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:15]
            names = [s[0] for s in sorted_scores][::-1]
            vals = [s[1] for s in sorted_scores][::-1]
            ax.barh(names, vals, color='#4CAF50')
        ax.set_xlabel(imp_type.capitalize())
        ax.set_title(f'XGBoost — {imp_type.capitalize()} Importance')
    fig.tight_layout()
    fig.savefig(f'{plots_dir}/xgb_feature_importance.png', dpi=120)
    plt.show()
    print('XGBoost provides 3 importance types:')
    print('  weight — how many times a feature is used for splits')
    print('  gain   — average improvement from splits on this feature')
    print('  cover  — average number of samples affected per split')

## 6 — Random Forest Deep Dive

Random Forest is the "reliable workhorse" — hard to overfit, works well
out of the box, and trivially parallelizable. It builds many full-depth
trees on bootstrap samples and averages their predictions.

Key points that differentiate RF from boosting:

1. **No learning rate** — each tree is independent; no sequential correction
2. **OOB (Out-of-Bag) score** — free generalization estimate without a
   holdout set. Each tree is trained on ~63.2% of the data (bootstrap), and
   validated on the remaining ~36.8%. This is unique to bagging methods.
3. **`max_features`** — fraction of features considered at each split. This
   is the primary regularization knob (more important than `max_depth`).
4. **`class_weight='balanced'`** — handles imbalanced classes by adjusting
   sample weights inversely proportional to class frequencies.
5. **Embarrassingly parallel** — set `n_jobs=-1` to use all CPU cores.

Key hyperparameters tuned:
- `n_estimators` — number of trees
- `max_depth` — maximum tree depth
- `min_samples_split`, `min_samples_leaf` — split/leaf constraints
- `max_features` — fraction of features per split

In [ ]:
# ---------------------------------------------------------------------------
# Random Forest — Optuna hyperparameter tuning + final model
# ---------------------------------------------------------------------------


def rf_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'max_depth': trial.suggest_int('max_depth', 3, 25),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features': trial.suggest_float('max_features', 0.2, 1.0),
        'class_weight': 'balanced',
    }
    metric_fn = _METRIC_FN[optimize_metric]
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=random_state)
    fold_scores = []
    for tr_idx, vl_idx in skf.split(X_train_np, y_train):
        model = RandomForestClassifier(
            **params, random_state=random_state, n_jobs=-1,
        )
        model.fit(X_train_np[tr_idx], y_train[tr_idx])
        proba = model.predict_proba(X_train_np[vl_idx])[:, 1]
        best = max(
            metric_fn(y_train[vl_idx], (proba >= t).astype(int), zero_division=0)
            for t in threshold_grid
        )
        fold_scores.append(best)
    return float(np.mean(fold_scores))


print('Tuning Random Forest...')
rf_study = optuna.create_study(direction='maximize')
rf_study.optimize(rf_objective, n_trials=optuna_n_trials, timeout=optuna_timeout_s)

rf_best_params = rf_study.best_params
print(f'Best CV {optimize_metric}: {rf_study.best_value:.4f}')
print(f'Best params: {json.dumps(rf_best_params, indent=2)}')

# Train final Random Forest model
print('\nTraining final Random Forest model...')
rf_model = RandomForestClassifier(
    **rf_best_params, class_weight='balanced',
    random_state=random_state, n_jobs=-1, oob_score=True,
)
rf_t0 = time.time()
rf_model.fit(X_train_np, y_train)
rf_train_time = time.time() - rf_t0

joblib.dump(rf_model, 'outputs/models/random_forest.joblib')
print(f'\nRandom Forest training time: {rf_train_time:.2f}s')
print(f'OOB score: {rf_model.oob_score_:.4f}')
print(f'Model saved -> outputs/models/random_forest.joblib')

In [ ]:
# ---------------------------------------------------------------------------
# Random Forest — bells & whistles
# ---------------------------------------------------------------------------

# -- OOB error vs n_estimators (convergence curve) -------------------------
# Use warm_start to incrementally add trees and track OOB score
print('Computing OOB convergence curve (warm_start)...')
rf_params_no_n = {k: v for k, v in rf_best_params.items() if k != 'n_estimators'}
n_max = rf_best_params['n_estimators']
n_range = list(range(10, n_max + 1, max(10, n_max // 30)))
if n_range[-1] != n_max:
    n_range.append(n_max)

oob_scores = []
rf_warm = RandomForestClassifier(
    **rf_params_no_n, n_estimators=1, warm_start=True, oob_score=True,
    class_weight='balanced', random_state=random_state, n_jobs=-1,
)
for n in n_range:
    rf_warm.set_params(n_estimators=n)
    rf_warm.fit(X_train_np, y_train)
    oob_scores.append(rf_warm.oob_score_)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(n_range, oob_scores, marker='.', linewidth=2, color='#2196F3')
ax.set_xlabel('Number of Trees')
ax.set_ylabel('OOB Accuracy')
ax.set_title('Random Forest — OOB Score Convergence')
ax.grid(True, alpha=0.3)
ax.axhline(rf_model.oob_score_, color='red', linestyle='--', alpha=0.7,
           label=f'Final OOB ({rf_model.oob_score_:.4f})')
ax.legend()
fig.tight_layout()
fig.savefig(f'{plots_dir}/rf_oob_convergence.png', dpi=120)
plt.show()
print(f'OOB score converges around {n_range[np.argmax(oob_scores)]} trees')

# -- Gini importance vs Permutation importance -----------------------------
if enable_feature_importance:
    from sklearn.inspection import permutation_importance

    gini_imp = rf_model.feature_importances_
    perm_result = permutation_importance(
        rf_model, X_test_np, y_test, n_repeats=10,
        random_state=random_state, n_jobs=-1,
    )
    perm_imp = perm_result.importances_mean

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    for ax, imp, title, color in [
        (axes[0], gini_imp, 'Gini Impurity (Default)', '#2196F3'),
        (axes[1], perm_imp, 'Permutation Importance', '#1565C0'),
    ]:
        top_idx = np.argsort(imp)[-15:]
        ax.barh([feature_names[i] for i in top_idx], imp[top_idx], color=color)
        ax.set_xlabel('Importance')
        ax.set_title(f'Random Forest — {title}')
    fig.tight_layout()
    fig.savefig(f'{plots_dir}/rf_feature_importance.png', dpi=120)
    plt.show()
    print('Gini importance can be biased toward high-cardinality features.')
    print('Permutation importance (right) is more reliable — it measures')
    print('the drop in accuracy when a feature is shuffled.')

## 7 — Logistic Regression Deep Dive

Logistic Regression is the gold standard baseline. If a linear model
performs comparably to tree ensembles, you should strongly consider using
it — the interpretability and simplicity are invaluable.

Key differences from tree-based models:
1. **Linear decision boundary** — learns `w0 + w1*x1 + w2*x2 + ... = logit(p)`
2. **Requires feature scaling** — coefficients are meaningless without it
3. **Coefficients are directly interpretable** — positive coefficient means
   the feature increases the probability of the positive class
4. **Regularization types:**
   - **L2 (Ridge)** — shrinks all coefficients toward zero, never exactly zero
   - **L1 (Lasso)** — drives unimportant coefficients to exactly zero (feature selection!)
   - **ElasticNet** — mix of L1 and L2 (controlled by `l1_ratio`)
5. **`C` parameter** — inverse of regularization strength. Small C = strong
   regularization, large C = weak regularization.

In [ ]:
# ---------------------------------------------------------------------------
# Logistic Regression — Optuna hyperparameter tuning + final model
# ---------------------------------------------------------------------------


def lr_objective(trial):
    penalty = trial.suggest_categorical('penalty', ['l1', 'l2', 'elasticnet'])
    params = {
        'C': trial.suggest_float('C', 1e-4, 100.0, log=True),
        'penalty': penalty,
        'max_iter': 2000,
        'class_weight': 'balanced',
    }
    if penalty == 'elasticnet':
        params['l1_ratio'] = trial.suggest_float('l1_ratio', 0.1, 0.9)
        params['solver'] = 'saga'
    elif penalty == 'l1':
        params['solver'] = 'saga'
    else:
        params['solver'] = 'lbfgs'

    metric_fn = _METRIC_FN[optimize_metric]
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=random_state)
    fold_scores = []
    for tr_idx, vl_idx in skf.split(X_train_scaled, y_train):
        model = LogisticRegression(**params, random_state=random_state)
        model.fit(X_train_scaled[tr_idx], y_train[tr_idx])
        proba = model.predict_proba(X_train_scaled[vl_idx])[:, 1]
        best = max(
            metric_fn(y_train[vl_idx], (proba >= t).astype(int), zero_division=0)
            for t in threshold_grid
        )
        fold_scores.append(best)
    return float(np.mean(fold_scores))


print('Tuning Logistic Regression...')
lr_study = optuna.create_study(direction='maximize')
lr_study.optimize(lr_objective, n_trials=optuna_n_trials, timeout=optuna_timeout_s)

lr_best_params = lr_study.best_params
print(f'Best CV {optimize_metric}: {lr_study.best_value:.4f}')
print(f'Best params: {json.dumps(lr_best_params, indent=2)}')

# Train final Logistic Regression model
print('\nTraining final Logistic Regression model...')
lr_fit_params = {k: v for k, v in lr_best_params.items()}
lr_fit_params['max_iter'] = 2000
lr_fit_params['class_weight'] = 'balanced'
if lr_fit_params.get('penalty') == 'elasticnet':
    lr_fit_params['solver'] = 'saga'
elif lr_fit_params.get('penalty') == 'l1':
    lr_fit_params['solver'] = 'saga'
else:
    lr_fit_params['solver'] = 'lbfgs'

lr_model = LogisticRegression(**lr_fit_params, random_state=random_state)

lr_t0 = time.time()
lr_model.fit(X_train_scaled, y_train)
lr_train_time = time.time() - lr_t0

joblib.dump({'model': lr_model, 'scaler': scaler, 'encoder': oe},
            'outputs/models/logistic_regression.joblib')
print(f'\nLogistic Regression training time: {lr_train_time:.4f}s')
print(f'Model saved -> outputs/models/logistic_regression.joblib')
print(f'  (Saved with scaler + encoder — needed for inference preprocessing)')

In [ ]:
# ---------------------------------------------------------------------------
# Logistic Regression — bells & whistles
# ---------------------------------------------------------------------------

# -- Coefficient plot (interpretability!) ----------------------------------
coefs = lr_model.coef_[0]
coef_df = (
    pl.DataFrame({'feature': feature_names, 'coefficient': coefs})
    .sort('coefficient')
)

fig, ax = plt.subplots(figsize=(8, max(5, len(feature_names) * 0.35)))
colors_coef = ['#E53935' if c < 0 else '#4CAF50' for c in coef_df['coefficient'].to_list()]
ax.barh(coef_df['feature'].to_list(), coef_df['coefficient'].to_list(), color=colors_coef)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Coefficient (scaled features)')
ax.set_title('Logistic Regression — Coefficients\n(green = increases P(positive), red = decreases)')
ax.grid(True, alpha=0.3, axis='x')
fig.tight_layout()
fig.savefig(f'{plots_dir}/lr_coefficients.png', dpi=120)
plt.show()

n_nonzero = int(np.sum(np.abs(coefs) > 1e-6))
print(f'Non-zero coefficients: {n_nonzero} / {len(coefs)}')
print(f'Penalty: {lr_model.penalty}, C: {lr_model.C:.4f}')

# -- Regularization path: accuracy + sparsity vs C ------------------------
print('\nComputing regularization path...')
C_range = np.logspace(-4, 2, 25)
test_scores_l2 = []
test_scores_l1 = []
n_nonzero_l1 = []

for C_val in C_range:
    # L2
    lr_l2 = LogisticRegression(
        C=C_val, penalty='l2', solver='lbfgs', max_iter=2000,
        class_weight='balanced', random_state=random_state,
    )
    lr_l2.fit(X_train_scaled, y_train)
    test_scores_l2.append(lr_l2.score(X_test_scaled, y_test))

    # L1
    lr_l1 = LogisticRegression(
        C=C_val, penalty='l1', solver='saga', max_iter=2000,
        class_weight='balanced', random_state=random_state,
    )
    lr_l1.fit(X_train_scaled, y_train)
    test_scores_l1.append(lr_l1.score(X_test_scaled, y_test))
    n_nonzero_l1.append(int(np.sum(np.abs(lr_l1.coef_[0]) > 1e-6)))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.semilogx(C_range, test_scores_l2, 'o-', label='L2 (Ridge)', color='#2196F3')
ax.semilogx(C_range, test_scores_l1, 's-', label='L1 (Lasso)', color='#FF9800')
ax.set_xlabel('C (inverse regularization strength)')
ax.set_ylabel('Test Accuracy')
ax.set_title('Regularization Path — Accuracy vs C')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.semilogx(C_range, n_nonzero_l1, 's-', color='#FF9800')
ax.set_xlabel('C (inverse regularization strength)')
ax.set_ylabel('Non-zero Coefficients')
ax.set_title('L1 Sparsity — Feature Selection Effect')
ax.axhline(len(feature_names), color='gray', linestyle='--', alpha=0.5,
           label=f'Total features ({len(feature_names)})')
ax.legend()
ax.grid(True, alpha=0.3)

fig.tight_layout()
fig.savefig(f'{plots_dir}/lr_regularization_path.png', dpi=120)
plt.show()
print('Left: small C = strong regularization (underfitting), large C = weak (overfitting)')
print('Right: L1 drives coefficients to exactly zero — automatic feature selection')

## 8 — HistGradientBoosting Deep Dive

sklearn's `HistGradientBoostingClassifier` is the **no-extra-dependencies**
boosting option. It was inspired by LightGBM and added to sklearn in v0.21
(stabilized in v1.0). If you're already in the sklearn ecosystem and don't
want to add XGBoost/LightGBM/CatBoost as a dependency, this is your model.

Key features:
1. **Histogram-based splits** — bins continuous features into 256 bins for
   fast split finding (same idea as LightGBM/XGBoost `tree_method='hist'`)
2. **Native categorical support** — pass `categorical_features` (list of
   indices) and ordinal-encoded integer columns. The model finds optimal
   categorical splits without one-hot encoding.
3. **Native NaN handling** — missing values are routed to the child that
   minimizes the loss. No imputation needed.
4. **Early stopping** — built-in via `early_stopping=True` with an internal
   validation split (`validation_fraction`)
5. **`max_leaf_nodes`** — controls tree complexity (similar to LightGBM's
   `num_leaves`)

In [ ]:
# ---------------------------------------------------------------------------
# HistGradientBoosting — Optuna hyperparameter tuning + final model
# ---------------------------------------------------------------------------


def hgb_objective(trial):
    params = {
        'max_iter': trial.suggest_int('max_iter', 100, 1500),
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 5, 50),
        'max_leaf_nodes': trial.suggest_int('max_leaf_nodes', 15, 255),
        'l2_regularization': trial.suggest_float('l2_regularization', 1e-3, 10.0, log=True),
    }
    metric_fn = _METRIC_FN[optimize_metric]
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=random_state)
    fold_scores = []
    for tr_idx, vl_idx in skf.split(X_train_np, y_train):
        model = HistGradientBoostingClassifier(
            **params, categorical_features=cat_feature_indices,
            random_state=random_state, early_stopping=True,
            validation_fraction=0.15, n_iter_no_change=30,
        )
        model.fit(X_train_np[tr_idx], y_train[tr_idx])
        proba = model.predict_proba(X_train_np[vl_idx])[:, 1]
        best = max(
            metric_fn(y_train[vl_idx], (proba >= t).astype(int), zero_division=0)
            for t in threshold_grid
        )
        fold_scores.append(best)
    return float(np.mean(fold_scores))


print('Tuning HistGradientBoosting...')
hgb_study = optuna.create_study(direction='maximize')
hgb_study.optimize(hgb_objective, n_trials=optuna_n_trials, timeout=optuna_timeout_s)

hgb_best_params = hgb_study.best_params
print(f'Best CV {optimize_metric}: {hgb_study.best_value:.4f}')
print(f'Best params: {json.dumps(hgb_best_params, indent=2)}')

# Train final HistGBM model
print('\nTraining final HistGradientBoosting model...')
hgb_model = HistGradientBoostingClassifier(
    **hgb_best_params, categorical_features=cat_feature_indices,
    random_state=random_state, early_stopping=True,
    validation_fraction=0.15, n_iter_no_change=50,
)
hgb_t0 = time.time()
hgb_model.fit(X_train_np, y_train)
hgb_train_time = time.time() - hgb_t0

joblib.dump(hgb_model, 'outputs/models/histgbm_model.joblib')
print(f'\nHistGBM training time: {hgb_train_time:.2f}s')
print(f'Iterations used: {hgb_model.n_iter_}')
print(f'Model saved -> outputs/models/histgbm_model.joblib')

In [ ]:
# ---------------------------------------------------------------------------
# HistGradientBoosting — bells & whistles
# ---------------------------------------------------------------------------

# -- Learning curve (train_score_ / validation_score_) ---------------------
fig, ax = plt.subplots(figsize=(9, 5))
if hasattr(hgb_model, 'train_score_'):
    ax.plot(hgb_model.train_score_, label='Train', alpha=0.8)
if hasattr(hgb_model, 'validation_score_'):
    ax.plot(hgb_model.validation_score_, label='Validation', alpha=0.8)
    ax.axvline(hgb_model.n_iter_, color='red', linestyle='--', alpha=0.7,
               label=f'Stopped at iter {hgb_model.n_iter_}')
ax.set_xlabel('Iteration')
ax.set_ylabel('Score')
ax.set_title('HistGradientBoosting — Learning Curve')
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(f'{plots_dir}/hgb_learning_curve.png', dpi=120)
plt.show()

# -- Native NaN handling demo ---------------------------------------------
X_nan_demo = X_test_np[:5].copy()
X_nan_demo[0, 0] = np.nan
X_nan_demo[1, 2] = np.nan
try:
    preds_nan = hgb_model.predict_proba(X_nan_demo)[:, 1]
    print('HistGBM handles NaN natively — no imputation needed:')
    for i in range(5):
        nans = np.isnan(X_nan_demo[i]).sum()
        print(f'  Sample {i} (NaNs={nans}): P(positive)={preds_nan[i]:.4f}')
except Exception as e:
    print(f'NaN handling: {e}')

# -- Feature importance ----------------------------------------------------
if enable_feature_importance:
    from sklearn.inspection import permutation_importance as perm_imp_fn

    # HistGBM may not have feature_importances_ in all sklearn versions
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Permutation importance (always available)
    hgb_perm = perm_imp_fn(
        hgb_model, X_test_np, y_test, n_repeats=10,
        random_state=random_state, n_jobs=-1,
    )
    top_idx = np.argsort(hgb_perm.importances_mean)[-15:]
    axes[0].barh(
        [feature_names[i] for i in top_idx],
        hgb_perm.importances_mean[top_idx],
        xerr=hgb_perm.importances_std[top_idx],
        color='#FF9800',
    )
    axes[0].set_xlabel('Mean Accuracy Decrease')
    axes[0].set_title('HistGBM — Permutation Importance')

    # Tree-based importance if available
    if hasattr(hgb_model, 'feature_importances_'):
        hgb_fi = hgb_model.feature_importances_
        top_idx2 = np.argsort(hgb_fi)[-15:]
        axes[1].barh(
            [feature_names[i] for i in top_idx2],
            hgb_fi[top_idx2], color='#E65100',
        )
        axes[1].set_xlabel('Importance')
        axes[1].set_title('HistGBM — Tree-based Importance')
    else:
        axes[1].text(0.5, 0.5, 'feature_importances_\nnot available\nin this sklearn version',
                     transform=axes[1].transAxes, ha='center', va='center', fontsize=12)
        axes[1].set_title('HistGBM — Tree-based Importance')

    fig.tight_layout()
    fig.savefig(f'{plots_dir}/hgb_feature_importance.png', dpi=120)
    plt.show()

## 9 — Head-to-Head: Performance Comparison

Same data split, same Optuna budget, same threshold analysis — a fair fight
between a linear model, a bagging ensemble, and two gradient boosting
implementations.

In [ ]:
# ---------------------------------------------------------------------------
# Compute predictions and threshold metrics for all four models
# ---------------------------------------------------------------------------
models_eval = {
    'XGBoost': (xgb_model, X_test_xgb),
    'RandomForest': (rf_model, X_test_np),
    'LogisticReg': (lr_model, X_test_scaled),
    'HistGBM': (hgb_model, X_test_np),
}
colors = {
    'XGBoost': '#4CAF50',
    'RandomForest': '#2196F3',
    'LogisticReg': '#9C27B0',
    'HistGBM': '#FF9800',
}

results = {}
for name, (model, X_test_fmt) in models_eval.items():
    y_proba = model.predict_proba(X_test_fmt)[:, 1]
    rows = []
    for t in threshold_grid:
        y_pred_t = (y_proba >= t).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_test, y_pred_t, labels=[0, 1]).ravel()
        rows.append({
            'threshold': round(t, 4),
            'precision': round(precision_score(y_test, y_pred_t, zero_division=0), 4),
            'recall': round(recall_score(y_test, y_pred_t, zero_division=0), 4),
            'f1': round(f1_score(y_test, y_pred_t, zero_division=0), 4),
            'tp': int(tp), 'fp': int(fp), 'tn': int(tn), 'fn': int(fn),
        })
    thr_df = pl.DataFrame(rows)
    best_row = thr_df.sort(optimize_metric, descending=True).row(0, named=True)
    roc_auc_val = roc_auc_score(y_test, y_proba)
    pr_auc_val = average_precision_score(y_test, y_proba)
    results[name] = {
        'y_proba': y_proba,
        'threshold_df': thr_df,
        'threshold_rows': rows,
        'best_row': best_row,
        'best_threshold': best_row['threshold'],
        'roc_auc': roc_auc_val,
        'pr_auc': pr_auc_val,
    }
    print(f'{name:14s}  best_thr={best_row["threshold"]:.2f}  '
          f'F1={best_row["f1"]:.4f}  P={best_row["precision"]:.4f}  '
          f'R={best_row["recall"]:.4f}  ROC-AUC={roc_auc_val:.4f}  '
          f'PR-AUC={pr_auc_val:.4f}')

In [ ]:
# ---------------------------------------------------------------------------
# Training time comparison + confusion matrices
# ---------------------------------------------------------------------------
train_times = {
    'XGBoost': xgb_train_time, 'RandomForest': rf_train_time,
    'LogisticReg': lr_train_time, 'HistGBM': hgb_train_time,
}

fig, axes = plt.subplots(1, 5, figsize=(27, 5))

# Training time bar chart
ax = axes[0]
bars = ax.bar(train_times.keys(), train_times.values(),
              color=[colors[k] for k in train_times.keys()], edgecolor='white')
for bar, t in zip(bars, train_times.values()):
    label = f'{t:.1f}s' if t >= 0.1 else f'{t*1000:.0f}ms'
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            label, ha='center', va='bottom', fontsize=9)
ax.set_ylabel('Seconds')
ax.set_title('Training Time')
ax.grid(True, alpha=0.3, axis='y')
ax.tick_params(axis='x', rotation=30)

# Confusion matrices
for i, (name, res) in enumerate(results.items()):
    ax = axes[i + 1]
    y_pred = (res['y_proba'] >= res['best_threshold']).astype(int)
    ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=ax, cmap='Blues')
    ax.set_title(f'{name}\n(thr={res["best_threshold"]:.2f})')

fig.tight_layout()
fig.savefig(f'{plots_dir}/training_time_confusion.png', dpi=120)
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# ROC and PR curves overlaid
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ROC curves
ax = axes[0]
for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_test, res['y_proba'])
    ax.plot(fpr, tpr, linewidth=2, label=f'{name} (AUC={res["roc_auc"]:.4f})',
            color=colors[name])
ax.plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves')
ax.legend(loc='lower right', fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)

# PR curves
ax = axes[1]
prevalence = y_test.mean()
for name, res in results.items():
    pr_prec, pr_rec, _ = precision_recall_curve(y_test, res['y_proba'])
    ax.plot(pr_rec, pr_prec, linewidth=2, label=f'{name} (AP={res["pr_auc"]:.4f})',
            color=colors[name])
ax.axhline(prevalence, color='k', linestyle='--', alpha=0.4,
           label=f'Baseline ({prevalence:.3f})')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curves')
ax.legend(loc='upper right', fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.05)

fig.tight_layout()
fig.savefig(f'{plots_dir}/roc_pr_curves.png', dpi=120)
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Precision / Recall / F1 vs Threshold (one subplot per model)
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(1, 4, figsize=(24, 5))
for ax, (name, res) in zip(axes, results.items()):
    thr_df = res['threshold_df']
    thresholds = thr_df['threshold'].to_list()
    ax.plot(thresholds, thr_df['precision'].to_list(), label='Precision', marker='.', color='#4CAF50')
    ax.plot(thresholds, thr_df['recall'].to_list(), label='Recall', marker='.', color='#FF9800')
    ax.plot(thresholds, thr_df['f1'].to_list(), label='F1', marker='.', linewidth=2, color='#2196F3')
    ax.axvline(res['best_threshold'], color='red', linestyle='--', alpha=0.7,
               label=f'Best thr={res["best_threshold"]}')
    ax.set_xlabel('Threshold')
    ax.set_ylabel('Score')
    ax.set_title(f'{name}')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(-0.05, 1.05)
fig.suptitle('Metrics vs Threshold', fontsize=14, y=1.02)
fig.tight_layout()
fig.savefig(f'{plots_dir}/threshold_per_model.png', dpi=120, bbox_inches='tight')
plt.show()

# ---------------------------------------------------------------------------
# Calibration curves overlaid
# ---------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(7, 6))
for name, res in results.items():
    prob_true, prob_pred = calibration_curve(y_test, res['y_proba'], n_bins=10, strategy='uniform')
    ax.plot(prob_pred, prob_true, 's-', linewidth=2, label=name, color=colors[name])
ax.plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Perfectly calibrated')
ax.set_xlabel('Mean Predicted Probability')
ax.set_ylabel('Fraction of Positives')
ax.set_title('Calibration Curves\n(Logistic Regression is typically best-calibrated)')
ax.legend(loc='lower right', fontsize=9)
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(f'{plots_dir}/calibration_curves.png', dpi=120)
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Predicted probability distributions
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(1, 4, figsize=(22, 5))
mask_neg = y_test == 0
mask_pos = y_test == 1
for ax, (name, res) in zip(axes, results.items()):
    ax.hist(res['y_proba'][mask_neg], bins=50, alpha=0.6, label='Class 0',
            color='#2196F3', edgecolor='white')
    ax.hist(res['y_proba'][mask_pos], bins=50, alpha=0.6, label='Class 1',
            color='#FF5722', edgecolor='white')
    ax.axvline(res['best_threshold'], color='red', linestyle='--', linewidth=2,
               label=f'thr={res["best_threshold"]}')
    ax.set_xlabel('Predicted Probability')
    ax.set_ylabel('Count')
    ax.set_title(f'{name}')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)
fig.suptitle('Predicted Probability Distribution by Class', fontsize=14, y=1.02)
fig.tight_layout()
fig.savefig(f'{plots_dir}/probability_distributions.png', dpi=120, bbox_inches='tight')
plt.show()

# ---------------------------------------------------------------------------
# Summary comparison table
# ---------------------------------------------------------------------------
print('\n' + '=' * 100)
print(f'{"Metric":<25} {"XGBoost":>12} {"RandomForest":>14} {"LogisticReg":>13} {"HistGBM":>12}   {"Winner":>14}')
print('=' * 100)

for label, key in [('Best F1', 'f1'), ('Precision @ best thr', 'precision'), ('Recall @ best thr', 'recall')]:
    vals = {name: res['best_row'][key] for name, res in results.items()}
    winner = max(vals, key=vals.get)
    print(f'{label:<25} {vals["XGBoost"]:>12.4f} {vals["RandomForest"]:>14.4f} '
          f'{vals["LogisticReg"]:>13.4f} {vals["HistGBM"]:>12.4f}   {winner:>14}')

for label, key in [('ROC-AUC', 'roc_auc'), ('PR-AUC', 'pr_auc')]:
    vals = {name: res[key] for name, res in results.items()}
    winner = max(vals, key=vals.get)
    print(f'{label:<25} {vals["XGBoost"]:>12.4f} {vals["RandomForest"]:>14.4f} '
          f'{vals["LogisticReg"]:>13.4f} {vals["HistGBM"]:>12.4f}   {winner:>14}')

vals = train_times
winner = min(vals, key=vals.get)
print(f'{"Training Time (s)":<25} {vals["XGBoost"]:>12.2f} {vals["RandomForest"]:>14.2f} '
      f'{vals["LogisticReg"]:>13.4f} {vals["HistGBM"]:>12.2f}   {winner:>14}')

best_thrs = {name: res['best_threshold'] for name, res in results.items()}
print(f'{"Best Threshold":<25} {best_thrs["XGBoost"]:>12.2f} {best_thrs["RandomForest"]:>14.2f} '
      f'{best_thrs["LogisticReg"]:>13.2f} {best_thrs["HistGBM"]:>12.2f}')
print('=' * 100)

## 10 — Feature Importance & SHAP Comparison

We compare what each model says about feature importance:
- **XGBoost:** TreeSHAP (exact, fast)
- **Random Forest:** TreeSHAP (exact for tree ensembles)
- **Logistic Regression:** Coefficient magnitudes (direct interpretability)
  plus LinearSHAP for comparable SHAP values
- **HistGradientBoosting:** TreeSHAP if supported, otherwise permutation
  importance

Do fundamentally different model types agree on which features matter?
Disagreements between the linear model and tree models reveal **non-linear
effects** that LogReg cannot capture.

In [ ]:
# ---------------------------------------------------------------------------
# Side-by-side feature importance (each model's native type)
# ---------------------------------------------------------------------------
if enable_feature_importance:
    fig, axes = plt.subplots(1, 4, figsize=(24, 7))

    # XGBoost — Gain (default for sklearn API)
    xgb_imp = xgb_model.feature_importances_
    xgb_imp_order = np.argsort(xgb_imp)[-15:]
    axes[0].barh([feature_names[i] for i in xgb_imp_order], xgb_imp[xgb_imp_order],
                 color=colors['XGBoost'])
    axes[0].set_title('XGBoost\n(Gain)')
    axes[0].set_xlabel('Importance')

    # Random Forest — Gini impurity
    rf_imp = rf_model.feature_importances_
    rf_imp_order = np.argsort(rf_imp)[-15:]
    axes[1].barh([feature_names[i] for i in rf_imp_order], rf_imp[rf_imp_order],
                 color=colors['RandomForest'])
    axes[1].set_title('Random Forest\n(Gini Impurity)')
    axes[1].set_xlabel('Importance')

    # Logistic Regression — absolute coefficients
    lr_imp = np.abs(lr_model.coef_[0])
    lr_imp_order = np.argsort(lr_imp)[-15:]
    axes[2].barh([feature_names[i] for i in lr_imp_order], lr_imp[lr_imp_order],
                 color=colors['LogisticReg'])
    axes[2].set_title('Logistic Reg.\n(|Coefficient|)')
    axes[2].set_xlabel('|Coefficient|')

    # HistGBM — permutation importance
    hgb_perm_imp = perm_imp_fn(
        hgb_model, X_test_np, y_test, n_repeats=10,
        random_state=random_state, n_jobs=-1,
    ).importances_mean
    hgb_imp_order = np.argsort(hgb_perm_imp)[-15:]
    axes[3].barh([feature_names[i] for i in hgb_imp_order], hgb_perm_imp[hgb_imp_order],
                 color=colors['HistGBM'])
    axes[3].set_title('HistGBM\n(Permutation)')
    axes[3].set_xlabel('Mean Accuracy Decrease')

    fig.suptitle('Feature Importance — Native Type per Model', fontsize=14, y=1.02)
    fig.tight_layout()
    fig.savefig(f'{plots_dir}/feature_importance_comparison.png', dpi=120, bbox_inches='tight')
    plt.show()

    # Rank correlation (tree models only, since LogReg importance is on scaled features)
    xgb_ranks = np.argsort(np.argsort(-xgb_imp))
    rf_ranks = np.argsort(np.argsort(-rf_imp))
    lr_ranks = np.argsort(np.argsort(-lr_imp))
    hgb_ranks = np.argsort(np.argsort(-hgb_perm_imp))

    print('Feature importance rank correlation (Spearman):')
    for n1, r1_arr in [('XGBoost', xgb_ranks), ('RandomForest', rf_ranks),
                        ('LogisticReg', lr_ranks), ('HistGBM', hgb_ranks)]:
        for n2, r2_arr in [('XGBoost', xgb_ranks), ('RandomForest', rf_ranks),
                            ('LogisticReg', lr_ranks), ('HistGBM', hgb_ranks)]:
            if n1 < n2:
                r, _ = spearmanr(r1_arr, r2_arr)
                print(f'  {n1:14s} vs {n2:14s}: {r:.3f}')
else:
    print('Feature importance disabled.')

In [ ]:
# ---------------------------------------------------------------------------
# SHAP comparison
# ---------------------------------------------------------------------------
if enable_shap:
    import shap

    rng_shap = np.random.RandomState(random_state)
    n_shap = min(shap_sample_size, X_test_np.shape[0])
    shap_idx = rng_shap.choice(X_test_np.shape[0], size=n_shap, replace=False)

    shap_results = {}

    # -- XGBoost SHAP (TreeExplainer) --------------------------------------
    print('Computing XGBoost SHAP values...')
    xgb_explainer = shap.TreeExplainer(xgb_model)
    xgb_sv = xgb_explainer.shap_values(X_test_xgb.iloc[shap_idx])
    shap_results['XGBoost'] = xgb_sv

    # -- Random Forest SHAP (TreeExplainer) --------------------------------
    print('Computing Random Forest SHAP values...')
    rf_explainer = shap.TreeExplainer(rf_model)
    rf_sv_raw = rf_explainer.shap_values(X_test_np[shap_idx])
    # RF binary classification returns [class0, class1]
    rf_sv = rf_sv_raw[1] if isinstance(rf_sv_raw, list) else rf_sv_raw
    shap_results['RandomForest'] = rf_sv

    # -- Logistic Regression SHAP (LinearExplainer) ------------------------
    print('Computing Logistic Regression SHAP values...')
    lr_explainer = shap.LinearExplainer(lr_model, X_train_scaled)
    lr_sv = lr_explainer.shap_values(X_test_scaled[shap_idx])
    shap_results['LogisticReg'] = lr_sv

    # -- HistGBM SHAP (TreeExplainer or fallback) --------------------------
    print('Computing HistGBM SHAP values...')
    try:
        hgb_explainer = shap.TreeExplainer(hgb_model)
        hgb_sv_raw = hgb_explainer.shap_values(X_test_np[shap_idx])
        hgb_sv = hgb_sv_raw[1] if isinstance(hgb_sv_raw, list) else hgb_sv_raw
        print('  -> Used TreeExplainer')
    except Exception:
        print('  -> TreeExplainer not supported, using Explainer (may be slower)...')
        hgb_explainer = shap.Explainer(hgb_model, X_train_np[:100])
        hgb_explanation = hgb_explainer(X_test_np[shap_idx[:min(100, n_shap)]])
        hgb_sv = hgb_explanation.values
        if hgb_sv.ndim == 3:
            hgb_sv = hgb_sv[:, :, 1]
    shap_results['HistGBM'] = hgb_sv

    # -- Summary bar plots (mean |SHAP|) -----------------------------------
    fig, axes = plt.subplots(1, 4, figsize=(24, 7))
    for ax, (name, sv) in zip(axes, shap_results.items()):
        mean_abs_shap = np.abs(sv).mean(axis=0)
        top_idx = np.argsort(mean_abs_shap)[-15:]
        ax.barh(
            [feature_names[i] for i in top_idx],
            mean_abs_shap[top_idx],
            color=colors[name],
        )
        ax.set_xlabel('mean |SHAP value|')
        ax.set_title(f'{name}')
    fig.suptitle('SHAP Feature Importance (mean |SHAP value|)', fontsize=14, y=1.02)
    fig.tight_layout()
    fig.savefig(f'{plots_dir}/shap_comparison.png', dpi=120, bbox_inches='tight')
    plt.show()

    # -- SHAP rank correlation ---------------------------------------------
    print('\nSHAP importance rank correlation (Spearman):')
    shap_imps = {name: np.abs(sv).mean(axis=0) for name, sv in shap_results.items()}
    for n1 in shap_results:
        for n2 in shap_results:
            if n1 < n2:
                r, _ = spearmanr(shap_imps[n1], shap_imps[n2])
                print(f'  {n1:14s} vs {n2:14s}: {r:.3f}')

    # -- Top-5 features per model ------------------------------------------
    print(f'\nTop-5 features by SHAP:')
    for name, imp in shap_imps.items():
        top5 = np.argsort(imp)[-5:][::-1]
        feats = ', '.join(feature_names[i] for i in top5)
        print(f'  {name:14s}: {feats}')
else:
    print('SHAP disabled.')

## 10.1 — Calibration: Which Model Has the Best Probability Estimates?

**Logistic Regression is theoretically calibrated** — its outputs are 
maximum-likelihood probability estimates under a logistic link function.
Tree ensembles tend to be **overconfident** — their probability outputs 
cluster near 0 and 1 because leaf values represent deterministic class 
predictions that get averaged.

A **reliability diagram** plots:
- Mean predicted probability per bin (x-axis) vs fraction of positives (y-axis)
- Perfect calibration: points lie on the diagonal
- The Expected Calibration Error (ECE) quantifies total miscalibration

This comparison is uniquely educational because it shows:
- **Why LR's probabilities can be used directly** as business risk scores
- **Why RF and tree models need Platt scaling** before being trusted as probabilities
- **HistGBM's calibration** — it inherits similar biases to XGBoost

In [ ]:
# ---------------------------------------------------------------------------
# Reliability Diagrams — Calibration of All Four Models
# ---------------------------------------------------------------------------
from sklearn.calibration import calibration_curve, CalibratedClassifierCV

n_bins_cal = 10
model_names_all = ['XGBoost', 'RandomForest', 'LogisticReg', 'HistGBM']

fig, axes = plt.subplots(1, 4, figsize=(22, 5))

ece_scores = {}
for ax, name in zip(axes, model_names_all):
    y_proba = results[name]['y_proba']

    # Calibration curve with quantile strategy (equal sample count per bin)
    prob_true, prob_pred = calibration_curve(y_test, y_proba, n_bins=n_bins_cal,
                                              strategy='quantile')

    # Expected Calibration Error
    bin_edges = np.percentile(y_proba, np.linspace(0, 100, n_bins_cal + 1))
    bin_edges[-1] += 1e-8
    bin_counts = np.histogram(y_proba, bins=bin_edges)[0]
    weights = bin_counts / max(bin_counts.sum(), 1)
    n = min(len(prob_true), len(prob_pred), len(weights))
    ece = float(np.sum(weights[:n] * np.abs(prob_true[:n] - prob_pred[:n])))
    ece_scores[name] = ece

    ax.plot(prob_pred, prob_true, 'o-', color=colors[name], linewidth=2, markersize=7,
            label=f'ECE = {ece:.4f}', zorder=3)
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, linewidth=1, label='Perfect')
    ax.fill_between(prob_pred, prob_pred, prob_true, alpha=0.1, color=colors[name])

    # Rug plot showing probability distribution
    ax2 = ax.twinx()
    ax2.hist(y_proba[y_test == 0], bins=25, alpha=0.15, color='steelblue', density=True)
    ax2.hist(y_proba[y_test == 1], bins=25, alpha=0.15, color='tomato', density=True)
    ax2.set_yticks([])

    ax.set_xlabel('Mean Predicted Probability')
    ax.set_ylabel('Fraction of Positives')
    ax.set_title(f'{name}\nECE={ece:.4f}', fontsize=11)
    ax.legend(fontsize=9, loc='upper left')
    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.02, 1.02)
    ax.grid(True, alpha=0.3)

fig.suptitle('Reliability Diagrams — Calibration Quality\n'
             'Diagonal = perfect calibration; ECE = Expected Calibration Error (lower is better)',
             fontsize=12, y=1.01)
plt.tight_layout()
_cal_path = os.path.join(plots_dir, 'calibration_comparison.png')
fig.savefig(_cal_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved: {_cal_path}')
print('\nCalibration rankings (lower ECE = better calibrated):')
for name, ece in sorted(ece_scores.items(), key=lambda x: x[1]):
    star = ' ← best' if name == min(ece_scores, key=ece_scores.get) else ''
    print(f'  {name:<14}: ECE = {ece:.4f}{star}')
print('\nExpected: LogisticReg should have lowest ECE (theoretically calibrated).')
print('Tree models often need post-hoc calibration via Platt scaling or isotonic regression.')

## 10.2 — Learning Curves: How Much Data Does Each Model Need?

**Learning curves** plot model performance as a function of training set size. 
They answer the most common question in ML: *"Would more data help?"*

Patterns to recognize:
- **High bias (underfitting):** both train and val curves plateau at low performance → model is too simple, more data won't help, need more features or a more complex model
- **High variance (overfitting):** train score >> val score, val score still rising → model is too complex, more data *will* help
- **Sweet spot:** train ≈ val at high performance → well-fitted model

Logistic Regression has **high bias/low variance**: it learns quickly from a few 
samples but may plateau early. Random Forest and boosting models have **low bias/
higher variance**: they improve more with additional data but need more to converge.

The crossover point — where a simpler model would have needed far less data to 
achieve the same performance — reveals the **sample efficiency** of each algorithm.

In [ ]:
# ---------------------------------------------------------------------------
# Learning Curves — F1 vs Training Size
# ---------------------------------------------------------------------------
from sklearn.model_selection import learning_curve

# Combine train and test for learning curve evaluation
X_all = np.vstack([X_test_np, X_test_np])  # use test as proxy — or use actual train+test
# Actually build from X_train_pd + X_test_pd
X_lc_np = np.vstack([X_train_pd.values, X_test_pd.values])
y_lc = np.concatenate([y_train, y_test])

# Define models for learning curve (fresh, not the trained ones)
lc_models = {
    'LogisticReg':  LogisticRegression(max_iter=1000, random_state=random_state, C=1.0),
    'RandomForest': RandomForestClassifier(n_estimators=100, random_state=random_state, n_jobs=-1),
    'HistGBM':      HistGradientBoostingClassifier(max_iter=100, random_state=random_state),
}

train_sizes_frac = np.linspace(0.1, 1.0, 8)
cv_lc = 3  # 3-fold CV for speed

fig, axes = plt.subplots(1, len(lc_models), figsize=(18, 5))

for ax, (name, model) in zip(axes, lc_models.items()):
    try:
        train_sizes, train_scores, val_scores = learning_curve(
            model, X_lc_np, y_lc,
            train_sizes=train_sizes_frac,
            cv=cv_lc,
            scoring='f1',
            n_jobs=-1,
            shuffle=True,
            random_state=random_state,
        )

        train_mean = train_scores.mean(axis=1)
        train_std  = train_scores.std(axis=1)
        val_mean   = val_scores.mean(axis=1)
        val_std    = val_scores.std(axis=1)

        ax.plot(train_sizes, train_mean, 'o-', color=colors.get(name, 'steelblue'),
                linewidth=2, markersize=7, label='Train F1')
        ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std,
                        alpha=0.15, color=colors.get(name, 'steelblue'))
        ax.plot(train_sizes, val_mean, 's--', color='tomato',
                linewidth=2, markersize=7, label='Val F1 (CV)')
        ax.fill_between(train_sizes, val_mean - val_std, val_mean + val_std,
                        alpha=0.15, color='tomato')

        # Bias-variance diagnosis
        final_gap = float(train_mean[-1] - val_mean[-1])
        still_rising = float(val_mean[-1] - val_mean[-2]) > 0.005 if len(val_mean) > 1 else False
        diag = 'High variance' if final_gap > 0.05 else ('High bias' if val_mean[-1] < 0.6 else 'Well fitted')

        ax.set_xlabel('Training Set Size')
        ax.set_ylabel('F1 Score')
        ax.set_title(f'{name} Learning Curve\n[{diag}: gap={final_gap:.3f}]', fontsize=11)
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)
        ax.set_ylim(0, 1.05)
    except Exception as e:
        ax.text(0.5, 0.5, f'Error: {e}', ha='center', va='center', transform=ax.transAxes)

fig.suptitle('Learning Curves — F1 Score vs Training Size\n'
             'Converging curves = well-fitted; train >> val = overfitting; both low = underfitting',
             fontsize=12, y=1.01)
plt.tight_layout()
_lc_path = os.path.join(plots_dir, 'learning_curves.png')
fig.savefig(_lc_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved: {_lc_path}')

## 10.3 — Decision Boundary Visualization (PCA 2D Projection)

All four models learn different **decision boundaries** in feature space. 
Visualizing these boundaries on a 2D PCA projection reveals the fundamental 
differences in model inductive biases:

- **Logistic Regression:** linear boundary (hyperplane in high-D, line in 2D)  
  Even a "curved" 2D boundary is actually linear in the original feature space.
- **Random Forest:** step-function boundaries — axis-aligned rectangular regions 
  (because each tree splits on one feature at a time)
- **HistGBM/XGBoost:** smoother boundaries than RF due to gradient optimization; 
  more complex but still based on axis-aligned splits

**Caveat:** the PCA projection discards information — a boundary that looks 
linear in 2D may be non-linear in the full feature space. Use this as an  
intuition-building tool, not a definitive accuracy measure.

In [ ]:
# ---------------------------------------------------------------------------
# Decision Boundary Visualization — PCA 2D Projection
# ---------------------------------------------------------------------------
from sklearn.decomposition import PCA

# Fit PCA on training data
pca_db = PCA(n_components=2, random_state=random_state)
X_train_2d = pca_db.fit_transform(X_train_pd.values)
X_test_2d  = pca_db.transform(X_test_pd.values)
explained = pca_db.explained_variance_ratio_
print(f'PCA explained variance: {100 * explained.sum():.1f}% ({explained[0]:.1%} + {explained[1]:.1%})')

# Create 2D classifiers trained on PCA projection
lc_models_2d = {
    'LogisticReg':  LogisticRegression(max_iter=1000, random_state=random_state, C=1.0),
    'RandomForest': RandomForestClassifier(n_estimators=100, random_state=random_state, n_jobs=-1),
    'HistGBM':      HistGradientBoostingClassifier(max_iter=100, random_state=random_state),
    'XGBoost':      type(xgb_model)(n_estimators=100, random_state=random_state,
                                     verbosity=0, n_jobs=-1, eval_metric='logloss'),
}
for name, m2d in lc_models_2d.items():
    m2d.fit(X_train_2d, y_train)

# Create meshgrid for decision boundary
x_min, x_max = X_test_2d[:, 0].min() - 0.5, X_test_2d[:, 0].max() + 0.5
y_min, y_max = X_test_2d[:, 1].min() - 0.5, X_test_2d[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))
grid = np.c_[xx.ravel(), yy.ravel()]

fig, axes = plt.subplots(1, 4, figsize=(24, 5))
scatter_kw = dict(s=10, alpha=0.5, linewidths=0)

for ax, (name, m2d) in zip(axes, lc_models_2d.items()):
    # Get probability on grid
    try:
        Z = m2d.predict_proba(grid)[:, 1].reshape(xx.shape)
    except Exception:
        Z = m2d.predict(grid).reshape(xx.shape).astype(float)

    # Background color = probability
    ax.contourf(xx, yy, Z, levels=50, cmap='RdYlBu_r', alpha=0.7, vmin=0, vmax=1)
    ax.contour(xx, yy, Z, levels=[0.5], colors='black', linewidths=1.5, linestyles='-')

    # Test points
    mask0 = y_test == 0
    mask1 = y_test == 1
    ax.scatter(X_test_2d[mask0, 0], X_test_2d[mask0, 1],
               c='steelblue', label='Class 0', **scatter_kw)
    ax.scatter(X_test_2d[mask1, 0], X_test_2d[mask1, 1],
               c='tomato', label='Class 1', **scatter_kw, marker='^')

    # F1 on the 2D model
    y_pred_2d = m2d.predict(X_test_2d)
    f1_2d = f1_score(y_test, y_pred_2d)
    ax.set_title(f'{name}\n2D F1={f1_2d:.3f} (projection only!)', fontsize=10)
    ax.set_xlabel(f'PCA-1 ({explained[0]:.1%} var)')
    ax.set_ylabel(f'PCA-2 ({explained[1]:.1%} var)')
    ax.legend(fontsize=7, markerscale=2)
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)

fig.suptitle(f'Decision Boundaries on PCA 2D Projection\n'
             f'(Black line = 0.5 decision boundary; 2D F1 ≠ full-dimensional F1 — '
             f'PCA captures {100*explained.sum():.0f}% of variance)',
             fontsize=11, y=1.01)
plt.tight_layout()
_db_path = os.path.join(plots_dir, 'decision_boundaries.png')
fig.savefig(_db_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved: {_db_path}')
print('\nKey visual lesson:')
print('  LogisticReg: smooth diagonal boundary (linear in 2D = linear in all dims)')
print('  RandomForest/HistGBM: step-function boundaries (axis-aligned splits)')
print('  More complex = can fit training data better, but not always better on test')

## 11 — Practical Considerations & When to Use Which

### The Interpretability vs Performance Spectrum

These four models sit at different points on the interpretability-performance
curve:

```
More Interpretable                           More Powerful
       |                                           |
  LogisticReg -----> RandomForest -----> HistGBM -----> XGBoost
       |                  |                 |              |
  Coefficients       Gini/Perm          TreeSHAP       TreeSHAP
  Linear only       Non-linear         Non-linear     Non-linear
  Fast training     Parallel           Fast           Moderate
```

### Preprocessing Requirements

| Model | Scaling? | Categorical Encoding | Missing Values |
|---|---|---|---|
| XGBoost | No | `enable_categorical` (native) | Built-in |
| Random Forest | No | OrdinalEncoder or OneHot | Needs imputation |
| Logistic Reg. | **Yes** (critical) | OrdinalEncoder or OneHot | Needs imputation |
| HistGBM | No | OrdinalEncoder + `categorical_features` | Built-in |

### Deployment Complexity

- **Logistic Regression:** Simplest to deploy — just store coefficients,
  scaler params, and encoder. Can even be implemented as a SQL query.
- **Random Forest:** Save with joblib. Moderate size (many trees). Pure sklearn.
- **HistGBM:** Save with joblib. Pure sklearn — no external dependencies at
  inference time.
- **XGBoost:** Save as `.ubj`. Requires xgboost library at inference. But
  with `enable_categorical`, handles raw categoricals (with Categorical dtype).

### When to Use Which

| Scenario | Recommendation |
|---|---|
| Need to explain every prediction | Logistic Regression (coefficients) |
| Quick baseline (first model to try) | Logistic Regression (< 1 second) |
| Robust with minimal hyperparameter tuning | Random Forest |
| Maximum predictive performance | XGBoost or HistGBM |
| No external dependencies allowed | HistGBM (pure sklearn) |
| Data has missing values | HistGBM (native NaN handling) |
| Small dataset (< 1K rows) | Logistic Regression or Random Forest |
| Very large dataset (> 1M rows) | HistGBM (fastest boosting) |
| Feature selection needed | Logistic Regression with L1 |
| Non-linear feature interactions | Any tree model (RF, XGBoost, HistGBM) |
| Regulatory/compliance requirements | Logistic Regression |
| Ensembling / stacking | All four as diverse base learners |

### Key Insight: When Does the Linear Model Lose?

The gap between Logistic Regression and tree models reveals **how non-linear
your problem is**. If LogReg performs within 1-2% AUC of the tree models,
your features already capture the relevant signal linearly — use LogReg for
simplicity. If the gap is large (> 5% AUC), the tree models are capturing
non-linear effects and interactions that LogReg cannot.

### The Verdict

Start with Logistic Regression. If it performs well enough, ship it — the
interpretability and simplicity are worth the small performance cost. If you
need more power, HistGBM gives you boosting without external dependencies.
If you want the most mature boosting implementation with native categorical
support, use XGBoost. Random Forest is the insurance policy — robust, hard
to overfit, and great for ensembling with boosting models.

## 12 — Metrics JSON Output

In [ ]:
# ---------------------------------------------------------------------------
# Structured metrics JSON — all four models
# ---------------------------------------------------------------------------
metrics_output = {
    'run_metadata': {
        'timestamp': datetime.now(timezone.utc).isoformat(),
        'target_col': target_col,
        'test_size': test_size,
        'random_state': random_state,
        'optuna_n_trials': optuna_n_trials,
        'optimize_metric': optimize_metric,
        'n_train': int(X_train_np.shape[0]),
        'n_test': int(X_test_np.shape[0]),
        'n_features': len(feature_names),
        'feature_names': feature_names,
        'cat_features': cat_cols,
        'num_features': num_cols,
        'scale_pos_weight': round(scale_pos_weight, 4),
    },
    'models': {},
}

model_meta = [
    ('xgboost', 'XGBoost', xgb_study, xgb_train_time, xgb_best_params),
    ('random_forest', 'RandomForest', rf_study, rf_train_time, rf_best_params),
    ('logistic_regression', 'LogisticReg', lr_study, lr_train_time, lr_best_params),
    ('histgbm', 'HistGBM', hgb_study, hgb_train_time, hgb_best_params),
]

for key, display_name, study, train_t, best_params in model_meta:
    res = results[display_name]
    entry = {
        'best_optuna_cv_score': round(study.best_value, 4),
        'best_params': best_params,
        'training_time_s': round(train_t, 4),
        'best_threshold': res['best_threshold'],
        'roc_auc': round(res['roc_auc'], 4),
        'pr_auc': round(res['pr_auc'], 4),
        'best_threshold_metrics': res['best_row'],
        'per_threshold': res['threshold_rows'],
    }
    if display_name == 'RandomForest':
        entry['oob_score'] = round(rf_model.oob_score_, 4)
    metrics_output['models'][key] = entry

Path(metrics_json_path).parent.mkdir(parents=True, exist_ok=True)
with open(metrics_json_path, 'w') as f:
    json.dump(metrics_output, f, indent=2, default=str)
print(f'Metrics JSON saved -> {metrics_json_path}')

## 13 — Summary

In [ ]:
# ---------------------------------------------------------------------------
# Final summary
# ---------------------------------------------------------------------------
print('=' * 75)
print('RUN COMPLETE — SKLEARN COMPARISON')
print('=' * 75)
print(f'  Metrics JSON:    {metrics_json_path}')
print(f'  Plots directory: {plots_dir}')
if executed_notebook_path:
    print(f'  Notebook:        {executed_notebook_path}')
print()
print(f'  {"Model":<14} {"F1":>8} {"ROC-AUC":>10} {"PR-AUC":>10} {"Time":>10}')
print(f'  {"-"*14} {"-"*8} {"-"*10} {"-"*10} {"-"*10}')
for name, res in results.items():
    t = train_times[name]
    t_str = f'{t:.1f}s' if t >= 0.1 else f'{t*1000:.0f}ms'
    print(f'  {name:<14} {res["best_row"]["f1"]:>8.4f} {res["roc_auc"]:>10.4f} '
          f'{res["pr_auc"]:>10.4f} {t_str:>10}')
print()

f1_winner = max(results, key=lambda n: results[n]['best_row']['f1'])
auc_winner = max(results, key=lambda n: results[n]['roc_auc'])
speed_winner = min(train_times, key=train_times.get)

print(f'  Best F1:        {f1_winner}')
print(f'  Best ROC-AUC:   {auc_winner}')
print(f'  Fastest:        {speed_winner}')

# Check if linear model is competitive
lr_auc = results['LogisticReg']['roc_auc']
best_auc = max(res['roc_auc'] for res in results.values())
gap = best_auc - lr_auc
print()
if gap < 0.02:
    print(f'  LogReg is within {gap:.4f} AUC of the best model.')
    print(f'  -> Consider using LogReg for its simplicity and interpretability!')
else:
    print(f'  LogReg trails the best model by {gap:.4f} AUC.')
    print(f'  -> Tree models are capturing meaningful non-linear effects.')
print('=' * 75)